# Drive Mount Karo

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Path Set Karo

In [3]:
import os

# base path set
BASE_PATH = '/content/drive/MyDrive/Capstone Project/Flipkart_Sales_Project'
RAW_DATA  = f'{BASE_PATH}/1_Data/raw'

# Check karo files hain ya nahi
files = os.listdir(RAW_DATA)
print("Files found:")
for f in files:
    print(f" ✅ {f}")

Files found:
 ✅ olist_products_dataset.csv
 ✅ olist_order_reviews_dataset.csv
 ✅ olist_sellers_dataset.csv
 ✅ olist_orders_dataset.csv
 ✅ olist_geolocation_dataset.csv
 ✅ olist_customers_dataset.csv
 ✅ product_category_name_translation.csv
 ✅ olist_order_items_dataset.csv
 ✅ olist_order_payments_dataset.csv


#  Libraries Install + Import

In [8]:
# Install karo
'''!pip install pandas sqlalchemy psycopg2-binary prophet plotly -q'''

# Import karo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✅ Saari libraries load ho gayi!")

✅ Saari libraries load ho gayi!


# Data Load

In [9]:
# Saari files ek saath load karo
orders       = pd.read_csv(f'{RAW_DATA}/olist_orders_dataset.csv')
order_items  = pd.read_csv(f'{RAW_DATA}/olist_order_items_dataset.csv')
products     = pd.read_csv(f'{RAW_DATA}/olist_products_dataset.csv')
customers    = pd.read_csv(f'{RAW_DATA}/olist_customers_dataset.csv')
sellers      = pd.read_csv(f'{RAW_DATA}/olist_sellers_dataset.csv')
payments     = pd.read_csv(f'{RAW_DATA}/olist_order_payments_dataset.csv')
reviews      = pd.read_csv(f'{RAW_DATA}/olist_order_reviews_dataset.csv')

# Kitne records hain dekho
datasets = {
    'Orders'      : orders,
    'Order Items' : order_items,
    'Products'    : products,
    'Customers'   : customers,
    'Sellers'     : sellers,
    'Payments'    : payments,
    'Reviews'     : reviews
}

print("Dataset       | Rows    | Columns")
print("-" * 40)
for name, df in datasets.items():
    print(f"{name:<15}| {len(df):<8}| {df.shape[1]}")

Dataset       | Rows    | Columns
----------------------------------------
Orders         | 99441   | 8
Order Items    | 112650  | 7
Products       | 32951   | 9
Customers      | 99441   | 5
Sellers        | 3095    | 4
Payments       | 103886  | 5
Reviews        | 99224   | 7


# Quick Data Check

In [10]:
# Har dataset mein missing values kitne hain
print("=== MISSING VALUES CHECK ===\n")
for name, df in datasets.items():
    missing = df.isnull().sum().sum()
    print(f"{name:<15} → {missing} missing values")

=== MISSING VALUES CHECK ===

Orders          → 4908 missing values
Order Items     → 0 missing values
Products        → 2448 missing values
Customers       → 0 missing values
Sellers         → 0 missing values
Payments        → 0 missing values
Reviews         → 145903 missing values


# Data Cleaning

# Orders Clean

In [12]:
# Orders mein kya missing hai exactly
print("=== ORDERS - Column wise missing ===")
print(orders.isnull().sum())

=== ORDERS - Column wise missing ===
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


# Products Clean

In [13]:
print("=== PRODUCTS - Column wise missing ===")
print(products.isnull().sum())

=== PRODUCTS - Column wise missing ===
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


#  Reviews Clean

In [14]:
print("=== REVIEWS - Column wise missing ===")
print(reviews.isnull().sum())

=== REVIEWS - Column wise missing ===
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64


#  Data Cleaning

In [15]:
# =============================
# DATA CLEANING
# =============================

# --- ORDERS ---
# Delivered dates jo missing hain = undelivered orders
# Hum inhe drop nahi karenge, fillna karenge 'Not Available' se
orders['order_approved_at'] = orders['order_approved_at'].fillna('Not Available')
orders['order_delivered_carrier_date'] = orders['order_delivered_carrier_date'].fillna('Not Available')
orders['order_delivered_customer_date'] = orders['order_delivered_customer_date'].fillna('Not Available')

print("✅ Orders cleaned!")

# --- PRODUCTS ---
# Category missing = 'Unknown' se fill karo
products['product_category_name'] = products['product_category_name'].fillna('unknown')
# Baaki numeric columns — median se fill karo
products['product_name_lenght'] = products['product_name_lenght'].fillna(products['product_name_lenght'].median())
products['product_description_lenght'] = products['product_description_lenght'].fillna(products['product_description_lenght'].median())
products['product_photos_qty'] = products['product_photos_qty'].fillna(products['product_photos_qty'].median())
products['product_weight_g'] = products['product_weight_g'].fillna(products['product_weight_g'].median())
products['product_length_cm'] = products['product_length_cm'].fillna(products['product_length_cm'].median())
products['product_height_cm'] = products['product_height_cm'].fillna(products['product_height_cm'].median())
products['product_width_cm'] = products['product_width_cm'].fillna(products['product_width_cm'].median())

print("✅ Products cleaned!")

# --- REVIEWS ---
# Comment title aur message — log likhte nahi, 'No Comment' se fill karo
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('No Comment')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('No Comment')

print("✅ Reviews cleaned!")

✅ Orders cleaned!
✅ Products cleaned!
✅ Reviews cleaned!


#  Verify Data is Cleaned or not

In [16]:
print("=== CLEANING VERIFY ===\n")
for name, df in datasets.items():
    missing = df.isnull().sum().sum()
    status = "✅ Clean" if missing == 0 else f"❌ {missing} missing"
    print(f"{name:<15} → {status}")

=== CLEANING VERIFY ===

Orders          → ✅ Clean
Order Items     → ✅ Clean
Products        → ✅ Clean
Customers       → ✅ Clean
Sellers         → ✅ Clean
Payments        → ✅ Clean
Reviews         → ✅ Clean


# Creating Master Dataset

In [17]:
# Saare tables merge karo
master_df = order_items.merge(orders, on='order_id', how='left')
master_df = master_df.merge(products, on='product_id', how='left')
master_df = master_df.merge(customers, on='customer_id', how='left')
master_df = master_df.merge(payments, on='order_id', how='left')
master_df = master_df.merge(sellers, on='seller_id', how='left')

print(f"✅ Master Dataset ready!")
print(f"Total Rows    : {len(master_df)}")
print(f"Total Columns : {master_df.shape[1]}")
print(f"\nColumns list:")
print(master_df.columns.tolist())

✅ Master Dataset ready!
Total Rows    : 117604
Total Columns : 33

Columns list:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


#  Data Saved in Processed Folder

In [18]:
PROCESSED_PATH = f'{BASE_PATH}/1_Data/processed'

master_df.to_csv(f'{PROCESSED_PATH}/master_sales_data.csv', index=False)

print(f"✅ master_sales_data.csv saved!")
print(f"📁 Location: {PROCESSED_PATH}")

✅ master_sales_data.csv saved!
📁 Location: /content/drive/MyDrive/Capstone Project/Flipkart_Sales_Project/1_Data/processed
